### Manual: 00-intro

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


# Smoke test — pinned MIT BEVFusion (`da93b27`) + official fused checkpoint

Ticket 02 of the BEVFusion spray pilot. One end-to-end GPU run of the **pinned**
MIT BEVFusion on Kaggle T4:

1. build the pinned environment (Python 3.8 + torch 1.10.1+cu113 + mmcv-full 1.4.0 + mmdet 2.20.0,
   mmdet3d CUDA ops compiled from the `da93b27` tree);
2. build the detection model from the official config chain;
3. download the official fused checkpoint **inside the kernel** and load it;
4. report **missing / unexpected checkpoint keys**;
5. forward **one sample** and check every output is finite;
6. check the **depth tensor channel count** against what `DepthLSSTransform.dtransform` accepts.

## How to run

Cells are independent: after the bootstrap cell every cell can be re-run on its
own from the Kaggle UI (that is the reason for the cell-per-concern split — the
CLI can only run the whole kernel). Order:

| cell | concern | needs |
|---|---|---|
| bootstrap | writes the helper modules into `/kaggle/working/smoke_test/scripts` | – |
| system probe | GPU, driver, CUDA toolkits, gcc, dataset mount | bootstrap |
| venv 3.8 | `uv python install 3.8.20` + `uv venv` | bootstrap |
| pinned wheels | torch / mmcv-full / mmdet / numpy | venv |
| clone pin | `git clone` + `git checkout da93b27` + static evidence | – |
| cuda toolchain | pick a CUDA 11.x `nvcc` for the extension build | system probe |
| build ops | `python setup.py develop` + per-extension import + GPU micro-smoke | venv, clone, toolchain |
| checkpoint | download + sha256 | – |
| config | resolve the `${...}` config chain | clone |
| model | build BEVFusion | env, ops, config |
| checkpoint load | missing/unexpected keys | model, checkpoint |
| batch | input shapes (synthetic, config-shaped) | config |
| forward | finite check + depth-channel check | model, checkpoint, batch |
| report | aggregate pass/fail + artifacts | the above |

## Environment decisions (recorded, with reasons)

* **Pin `da93b27`**, not HEAD. At HEAD `BaseDepthTransform` grew `add_depth_features`
  / `height_expand` and feeds a 6-channel depth tensor into a `Conv2d(1, 8, 1)`.
  The release commit has neither flag (this notebook greps the clone to prove it).
* **Python 3.8.20 via `uv`**, because Kaggle's system Python is 3.13 and can never
  install torch 1.10.1 / mmcv-full 1.4.0 wheels (they are cp38 only).
* **torch 1.10.1+cu113** from `download.pytorch.org/whl/cu113` and
  **mmcv-full 1.4.0** from `download.openmmlab.com/mmcv/dist/cu113/torch1.10.0`,
  i.e. the exact pinned stack. Nothing is upgraded to make a newer CUDA work.
* **`nvcc` 11.x** is required: torch 1.10.1 raises when the compiling CUDA major
  differs from its own (`torch/utils/cpp_extension.py::_check_cuda_version`).
  Kaggle ships CUDA 12.x only, so the notebook pulls the CUDA 11.8 `nvcc` from
  NVIDIA's `ubuntu2204` apt repository (same apt key as the pre-configured
  `ubuntu2404` repo) and points `CUDA_HOME` at it.
* **torchpack is not installed.** The official entry points use it only to merge
  the config chain and interpolate `${...}`; `smoke_cfg.py` reproduces exactly
  that (root-first `default.yaml` merge, then eval of `${expr}`), because
  torchpack hard-depends on torchvision/tensorpack/tensorboard/scipy.

## Known deviations (all recorded in the artifacts)

* `https://bevfusion.mit.edu/files/pretrained/bevfusion-det.pth` — the URL named in
  the ticket — currently returns **HTTP 404** (the MIT site was rebuilt in 2026).
  The cell probes it first and records the failure, then downloads the same fused
  detection checkpoint from the link the upstream README lists today (Dropbox).
  Which URL served the file, its size and sha256 are recorded.
* The forward uses a **synthetic batch** with the config's shapes: the Kaggle
  mirror has no `nuscenes_infos_*.pkl` yet (open item of ticket 04), so no real
  sample can traverse the MIT pipeline. Nothing here is reported as a real-data
  result; `data_source = "synthetic"` is written into every artifact.

## Acceptance criteria (ticket 02)

- [ ] checkpoint loads; missing/unexpected keys are listed
- [ ] forward of one sample raises no shape error
- [ ] no NaN/Inf in any output
- [ ] depth tensor has exactly the channel count `dtransform` accepts
- [ ] results are saved into a report
